# 🇿🇼 TAPnPAY Fraud Detection v4.0 - Data Science Masterclass
### Comprehensive Analysis: EDA, Feature Engineering, & Neural-LightGBM Hybrid Scoring

---

## 📌 1. Project Introduction
In the Zimbabwe 2026 mobile money landscape, fraud patterns have shifted from simple social engineering to sophisticated **SIM swap automation** and **Mule networks**. 

This notebook provides the E2E pipeline for training the v4.0 production model, featuring:
- **Advanced EDA** (Class separation analysis)
- **Feature Engineering** (Velocity and Behavioral indicators)
- **Imbalance Correction** (SMOTE-Tomek hybridization)
- **Model Interpretability** (Feature Importance & ROC-AUC)

---

### 🛠️ Dependencies & Global Configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import os
import json
import warnings
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score, 
    f1_score, roc_curve, precision_recall_curve, auc
)
from imblearn.combine import SMOTETomek
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (12, 8)

# Paths
DATA_PATH = '../data/TAPnPAY_fraud_enhanced.csv'
MODEL_PATH = '../model/fraud_detection_model_v4_zimbabwe.lgb'
META_PATH = '../model/model_metadata_v4_zimbabwe.json'

print("✅ Setup Complete: Data Science environment initialized.")

## 🔍 2. Exploratory Data Analysis (EDA)
Visualizing the fraud patterns to understand the 'Digital Signature' of Zimbabwean fraud.

In [ ]:
df = pd.read_csv(DATA_PATH)

# 1. Class Distribution
plt.figure(figsize=(6, 6))
df['fraud_label'].value_counts().plot.pie(autopct='%1.1f%%', colors=['#66b3ff','#ff9999'])
plt.title("Fraud vs. Legitimate Transaction Ratio")
plt.show()

# 2. Feature Correlation Heatmap
plt.figure(figsize=(12, 10))
numeric_df = df.select_dtypes(include=[np.number])
sns.heatmap(numeric_df.corr(), annot=False, cmap='coolwarm', linewidths=0.5)
plt.title("Feature Correlation Analysis")
plt.show()

# 3. Amount vs. Fraud Label
sns.boxplot(x='fraud_label', y='amount', data=df)
plt.yscale('log')
plt.title("Transaction Amount Distribution by Class (Log Scale)")
plt.show()

## ⚙️ 3. Advanced Feature Engineering
Mapping raw behavioral attributes to predictive features.

In [ ]:
features = [
    'sim_change_frequency', 'network_type',
    'new_device_login', 'time_since_login_seconds',
    'is_smurf_pattern', 'recent_cashins_24h', 'is_post_downtime',
    'receiver_risk_score', 'is_legit_merchant', 'is_mule_destination',
    'merchant_name_risk', 'Token_latency_seconds', 'geo_velocity_kmh',
    'distance_from_last_cashout_km', 'transaction_hour', 'is_night_transaction',
    'cashout_interval_hours', 'amount', 'transaction_type'
]

X = df[features].copy()
y = df['fraud_label'].copy()

# Label Encoding for Categorical Data
le = LabelEncoder()
cat_cols = X.select_dtypes(include=['object']).columns
for col in cat_cols:
    X[col] = le.fit_transform(X[col])
    print(f"Encoded {col}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
print("\n✅ Split: 80% Training | 20% Testing")

## ⚖️ 4. Class Balancing (SMOTE-Tomek)
We use **SMOTE-Tomek links** to not only oversample the minority class but also clean up the overlap between classes for cleaner decision boundaries.

In [ ]:
smt = SMOTETomek(random_state=42)
X_res, y_res = smt.fit_resample(X_train, y_train)

print(f"Original distribution: {np.bincount(y_train)}")
print(f"Balanced distribution: {np.bincount(y_res)}")

## 🚀 5. Model Training: LightGBM GBDT
Hyper-speed gradient boosting optimized for recall.

In [ ]:
lgb_train = lgb.Dataset(X_res, label=y_res)
lgb_eval = lgb.Dataset(X_test, label=y_test, reference=lgb_train)

params = {
    'objective': 'binary',
    'metric': ['auc', 'binary_logloss'],
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'is_unbalance': False, # Already balanced via SMOTE
    'verbose': -1
}

model = lgb.train(
    params,
    lgb_train,
    num_boost_round=1000,
    valid_sets=[lgb_train, lgb_eval],
    callbacks=[lgb.early_stopping(stopping_rounds=100)]
)

## 📈 6. Model Evaluation (PRO Metrics)
Visualizing ROC Curves and Confusion Matrices.

In [ ]:
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba >= 0.5).astype(int)

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC)')
plt.legend(loc="lower right")
plt.show()

# 2. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

# 3. Feature Importance
lgb.plot_importance(model, max_num_features=15, importance_type='split', title='Top Features Driving Fraud Detection')
plt.show()

## 💾 7. Production Model Serialization
Saving everything for the FastAPI endpoint.

In [ ]:
# Save model in native LightGBM format (.lgb instead of .txt to avoid confusion)
model.save_model(MODEL_PATH)

metadata = {
    "version": "4.0.0",
    "environment": "Production",
    "metrics": {
        "auc": round(roc_auc, 4),
        "f1": round(f1_score(y_test, y_pred), 4)
    },
    "features": features
}

with open(META_PATH, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f"✅ Model and Metadata saved to {os.path.dirname(MODEL_PATH)}")